# Feature Engineering v2 - House Price Prediction

This notebook improves the previous feature engineering pipeline.

Main goal:

- improve public generalization
- add leakage-safe target encoding
- add location-aware features
- add quality/location/size interactions
- create improved final datasets for modeling

Important:

- No model training is done here.
- Basic EDA is not repeated here.
- Missing handling/type correction/core FE from v1 are reused.

## 0. Project Setup

This section:

- imports libraries
- finds project root
- creates FE v2 folders
- defines global constants

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import json
import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 150)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
ID_COL = "Id"
TARGET_COL = "SalePrice"
LOG_TARGET_COL = "LogSalePrice"

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
def find_project_root_for_fe_v2(start_path=None):
    """
    Find project root by searching upward for required FE v1 outputs.
    """
    if start_path is None:
        start_path = Path.cwd()

    start_path = Path(start_path).resolve()
    candidate_roots = [start_path] + list(start_path.parents)

    required_files = [
        "fe_result/06_X_train_advanced_engineered.csv",
        "fe_result/06_X_test_advanced_engineered.csv",
        "fe_result/10_y_train_log_final.csv",
        "fe_result/10_y_train_raw_final.csv",
        "fe_result/10_train_ids_final.csv",
        "fe_result/10_test_ids_final.csv",
    ]

    for root in candidate_roots:
        if all((root / file).exists() for file in required_files):
            return root

    raise FileNotFoundError(
        "Could not find project root with required FE v1 outputs inside fe_result/."
    )


PROJECT_ROOT = find_project_root_for_fe_v2()

FE_RESULT_DIR = PROJECT_ROOT / "fe_result"
FE_REPORT_DIR = PROJECT_ROOT / "fe_report"

FE_RESULT_V2_DIR = PROJECT_ROOT / "fe_result_v2"
FE_REPORT_V2_DIR = PROJECT_ROOT / "fe_report_v2"
FE_PLOT_V2_DIR = FE_REPORT_V2_DIR / "plots"
FE_STEP_REPORT_V2_DIR = FE_REPORT_V2_DIR / "step_reports"
FE_LOG_V2_DIR = FE_REPORT_V2_DIR / "logs"

for folder in [
    FE_RESULT_V2_DIR,
    FE_REPORT_V2_DIR,
    FE_PLOT_V2_DIR,
    FE_STEP_REPORT_V2_DIR,
    FE_LOG_V2_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FE_RESULT_DIR:", FE_RESULT_DIR)
print("FE_RESULT_V2_DIR:", FE_RESULT_V2_DIR)
print("FE_REPORT_V2_DIR:", FE_REPORT_V2_DIR)

PROJECT_ROOT: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\House-Price-Prediction-Regression
FE_RESULT_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\House-Price-Prediction-Regression\fe_result
FE_RESULT_V2_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\House-Price-Prediction-Regression\fe_result_v2
FE_REPORT_V2_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\House-Price-Prediction-Regression\fe_report_v2


### FE v2 metadata

In [3]:
fe_v2_metadata = {
    "notebook": "Feature_Engineering_v2.ipynb",
    "base_source": "FE v1 advanced engineered dataset",
    "main_goal": "Improve public score using location-aware and target-encoded features",
    "target_transform": "log1p(SalePrice)",
    "model_training_done_here": False,
    "main_new_feature_groups": [
        "OOF target encoding",
        "Neighborhood price tier",
        "location-quality interactions",
        "location-size interactions",
        "quality-age interactions",
        "luxury/risk tail features",
        "linear-specific cleanup"
    ],
    "leakage_rule": "All train target encodings must be out-of-fold only."
}

with open(FE_LOG_V2_DIR / "00_fe_v2_metadata.json", "w") as f:
    json.dump(fe_v2_metadata, f, indent=4)

pd.DataFrame(
    [{"item": k, "value": str(v)} for k, v in fe_v2_metadata.items()]
).to_csv(
    FE_REPORT_V2_DIR / "00_fe_v2_metadata.csv",
    index=False
)

fe_v2_metadata

{'notebook': 'Feature_Engineering_v2.ipynb',
 'base_source': 'FE v1 advanced engineered dataset',
 'main_goal': 'Improve public score using location-aware and target-encoded features',
 'target_transform': 'log1p(SalePrice)',
 'model_training_done_here': False,
 'main_new_feature_groups': ['OOF target encoding',
  'Neighborhood price tier',
  'location-quality interactions',
  'location-size interactions',
  'quality-age interactions',
  'luxury/risk tail features',
  'linear-specific cleanup'],
 'leakage_rule': 'All train target encodings must be out-of-fold only.'}

## 1. Load FE v1 Advanced Dataset

FE v2 starts from the FE v1 advanced engineered dataset.

Reason:

- missing handling already done
- type correction already done
- core engineered features already created
- ordinal score features already created
- component and interaction features already created

We will add new v2 features before final encoding.

### Load FE v1 advanced outputs

In [4]:
# keep_default_na=False is important.
# It prevents absence labels such as "None" from becoming NaN again.

X_train_v2_base = pd.read_csv(
    FE_RESULT_DIR / "06_X_train_advanced_engineered.csv",
    keep_default_na=False
)

X_test_v2_base = pd.read_csv(
    FE_RESULT_DIR / "06_X_test_advanced_engineered.csv",
    keep_default_na=False
)

y_train_log_df = pd.read_csv(FE_RESULT_DIR / "10_y_train_log_final.csv")
y_train_raw_df = pd.read_csv(FE_RESULT_DIR / "10_y_train_raw_final.csv")

train_ids_df = pd.read_csv(FE_RESULT_DIR / "10_train_ids_final.csv")
test_ids_df = pd.read_csv(FE_RESULT_DIR / "10_test_ids_final.csv")

y_log_v2 = y_train_log_df[LOG_TARGET_COL].copy()
y_raw_v2 = y_train_raw_df[TARGET_COL].copy()

train_ids_v2 = train_ids_df[ID_COL].copy()
test_ids_v2 = test_ids_df[ID_COL].copy()

print("Loaded FE v1 advanced datasets:")
print("X_train_v2_base:", X_train_v2_base.shape)
print("X_test_v2_base:", X_test_v2_base.shape)
print("y_log_v2:", y_log_v2.shape)
print("y_raw_v2:", y_raw_v2.shape)
print("train_ids_v2:", train_ids_v2.shape)
print("test_ids_v2:", test_ids_v2.shape)

Loaded FE v1 advanced datasets:
X_train_v2_base: (1458, 137)
X_test_v2_base: (1459, 137)
y_log_v2: (1458,)
y_raw_v2: (1458,)
train_ids_v2: (1458,)
test_ids_v2: (1459,)


### Base validation

In [5]:
base_validation_report = pd.DataFrame([
    {
        "check": "train_rows_match_target",
        "value": X_train_v2_base.shape[0],
        "expected": len(y_log_v2),
        "status": "pass" if X_train_v2_base.shape[0] == len(y_log_v2) else "fail"
    },
    {
        "check": "test_rows_match_test_ids",
        "value": X_test_v2_base.shape[0],
        "expected": len(test_ids_v2),
        "status": "pass" if X_test_v2_base.shape[0] == len(test_ids_v2) else "fail"
    },
    {
        "check": "train_test_same_columns",
        "value": X_train_v2_base.columns.equals(X_test_v2_base.columns),
        "expected": True,
        "status": "pass" if X_train_v2_base.columns.equals(X_test_v2_base.columns) else "fail"
    },
    {
        "check": "target_not_in_train_features",
        "value": TARGET_COL in X_train_v2_base.columns,
        "expected": False,
        "status": "pass" if TARGET_COL not in X_train_v2_base.columns else "fail"
    },
    {
        "check": "target_not_in_test_features",
        "value": TARGET_COL in X_test_v2_base.columns,
        "expected": False,
        "status": "pass" if TARGET_COL not in X_test_v2_base.columns else "fail"
    },
    {
        "check": "id_column_available_train",
        "value": ID_COL in X_train_v2_base.columns,
        "expected": True,
        "status": "pass" if ID_COL in X_train_v2_base.columns else "warning"
    },
    {
        "check": "id_column_available_test",
        "value": ID_COL in X_test_v2_base.columns,
        "expected": True,
        "status": "pass" if ID_COL in X_test_v2_base.columns else "warning"
    },
])

base_validation_report.to_csv(
    FE_STEP_REPORT_V2_DIR / "01_base_dataset_validation_report.csv",
    index=False
)

base_validation_report

,check,value,expected,status
0,train_rows_match_target,1458,1458,pass
1,test_rows_match_test_ids,1459,1459,pass
2,train_test_same_columns,True,True,pass
3,target_not_in_train_features,False,False,pass
4,target_not_in_test_features,False,False,pass
5,id_column_available_train,True,True,pass
6,id_column_available_test,True,True,pass


### Stop if critical validation fails

In [6]:
critical_failed_checks = base_validation_report.loc[
    (base_validation_report["status"] == "fail")
]

if not critical_failed_checks.empty:
    display(critical_failed_checks)
    raise ValueError("FE v2 base validation failed. Fix before continuing.")

print("FE v2 base dataset validation passed.")

FE v2 base dataset validation passed.


### Inspect available important columns

In [7]:
important_v2_columns = [
    ID_COL,
    "Neighborhood",
    "MSSubClass",
    "OverallQual",
    "OverallCond",
    "GrLivArea",
    "TotalSF",
    "TotalBsmtSF",
    "1stFlrSF",
    "GarageArea",
    "GarageCars",
    "TotalBath",
    "HouseAge",
    "RemodAge",
    "GarageAge",
    "ExterQual_Ord",
    "KitchenQual_Ord",
    "BsmtQual_Ord",
    "GarageFinish_Ord",
    "GarageQual_Ord",
    "Foundation",
    "HouseStyle",
    "SaleCondition",
    "Exterior1st",
    "Exterior2nd",
    "GarageType",
]

important_column_check = pd.DataFrame([
    {
        "feature": col,
        "exists_in_train": col in X_train_v2_base.columns,
        "exists_in_test": col in X_test_v2_base.columns,
        "train_dtype": str(X_train_v2_base[col].dtype) if col in X_train_v2_base.columns else "missing",
        "test_dtype": str(X_test_v2_base[col].dtype) if col in X_test_v2_base.columns else "missing",
    }
    for col in important_v2_columns
])

important_column_check.to_csv(
    FE_STEP_REPORT_V2_DIR / "01_important_v2_column_check.csv",
    index=False
)

important_column_check

,feature,exists_in_train,exists_in_test,train_dtype,test_dtype
0,Id,True,True,int64,int64
1,Neighborhood,True,True,object,object
2,MSSubClass,True,True,int64,int64
3,OverallQual,True,True,int64,int64
4,OverallCond,True,True,int64,int64
5,GrLivArea,True,True,int64,int64
6,TotalSF,True,True,int64,float64
7,TotalBsmtSF,True,True,int64,float64
8,1stFlrSF,True,True,int64,int64
9,GarageArea,True,True,int64,float64


### Manual Decision

Feature Engineering v2 setup is complete.

Decisions:

- FE v2 starts from FE v1 advanced engineered dataset.
- FE v1 missing handling, type correction, core features, ordinal scores, component scores, and interaction features are reused.
- Final encoded datasets are not used as the starting point because FE v2 still needs categorical columns for target encoding.
- New outputs will be saved in `fe_result_v2` and `fe_report_v2`.
- No model training is performed in this notebook.

Next section:

`2. Leakage-Safe OOF Target Encoding`